In [1]:
# Cell 1
# Mount Drive and set baseline path
import os
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

def find_baseline_dir():
    candidates = [
        "/content/drive/MyDrive/final_project/baseline",
        "/content/drive/MyDrive/final_project/baseline/",
    ]
    for p in candidates:
        if os.path.isdir(p):
            return os.path.abspath(p)

    shared_root = "/content/drive/Shareddrives"
    if os.path.isdir(shared_root):
        for root, dirs, _ in os.walk(shared_root):
            if root.endswith("/final_project") and "baseline" in dirs:
                return os.path.abspath(os.path.join(root, "baseline"))

    raise FileNotFoundError("Could not find final_project/baseline in Drive.")

BASE_DIR = find_baseline_dir()
os.chdir(BASE_DIR)

print("BASE_DIR =", BASE_DIR)
print("CWD =", os.getcwd())

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
BASE_DIR = /content/drive/MyDrive/final_project/baseline
CWD = /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline
CUDA available: True
GPU: NVIDIA L4


In [2]:
# Cell 2
# Install KG deps
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel", "-q"])

subprocess.call(
    [sys.executable, "-m", "pip", "uninstall", "-y",
     "pyyaml", "tqdm", "numpy", "pandas", "networkx", "scikit-learn"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

pkgs = [
    "pyyaml==6.0.1",
    "tqdm==4.66.2",
    "numpy==1.26.4",
    "pandas==2.2.2",
    "networkx==3.3",
    "scikit-learn==1.4.2",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

import yaml, tqdm, numpy, pandas, networkx, sklearn

print("pyyaml:", yaml.__version__)
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("networkx:", networkx.__version__)
print("scikit-learn:", sklearn.__version__)
print("Installed OK")

pyyaml: 6.0.1
numpy: 1.26.4
pandas: 2.2.2
networkx: 3.3
scikit-learn: 1.4.2
Installed OK


In [3]:
# Cell 3
# Create dummy mlx_lm for Colab
import os

mlx_root = os.path.join(BASE_DIR, "mlx_lm")
os.makedirs(mlx_root, exist_ok=True)

with open(os.path.join(mlx_root, "__init__.py"), "w") as f:
    f.write("# dummy mlx_lm package for Colab\n")

with open(os.path.join(mlx_root, "utils.py"), "w") as f:
    f.write(
        "def convert(*args, **kwargs):\n"
        "    raise RuntimeError('mlx_lm.convert is not supported on this environment (dummy stub).')\n"
    )

print("Created dummy mlx_lm at:", mlx_root)

Created dummy mlx_lm at: /content/drive/MyDrive/final_project/baseline/mlx_lm


In [4]:
# Cell 4
# Clone or update repo
import os
import subprocess

REPO_URL = "https://github.com/ali-mohmmadi/KGP-CuriousLLM.git"
REPO_DIR = "/content/KGP-CuriousLLM"

if not os.path.isdir(REPO_DIR):
    print("Cloning repository into:", REPO_DIR)
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
else:
    print("Repo already exists. Pulling latest changes...")
    subprocess.check_call(["git", "-C", REPO_DIR, "pull"])

print("Repo ready at:", REPO_DIR)
print("Repo root files:", os.listdir(REPO_DIR)[:15])

Cloning repository into: /content/KGP-CuriousLLM
Repo ready at: /content/KGP-CuriousLLM
Repo root files: ['MDR_embedding_main.py', 'grid_search_mistral_main.py', '.git', 'kg_construct_main.py', 'ft_mistral_main.py', 'README.md', 'configs', 'requirements.txt', 'MDR_main.py', 'images', 'kgp_main.py', 'KGP', 'create_dirs.py', 'T5_main.py', 'quantize_mistral_main.py']


In [5]:
# Cell 5
# Add paths and import KG modules
import sys
import os
import json
import yaml
import pickle
import numpy as np
import torch

REPO_DIR = "/content/KGP-CuriousLLM"

if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from KGP.KG.neighbor_kg_construct import get_kg_graph_gpu, add_node_features
from KGP.LLMs.Mistral.quantize_mistral_mlx import load_config

print("Imports OK")
print("KGP package loaded from:", REPO_DIR)

Imports OK
KGP package loaded from: /content/KGP-CuriousLLM


In [6]:
#Cell 6
# Check 2Wiki embedding files
import os
import json
import numpy as np

EMB_RUN_ID = "2wikimultihopqa_dev2020wiki_1000_old"
EMB_DIR = os.path.join(BASE_DIR, "DATA", "KG", "emb", f"emb_{EMB_RUN_ID}")

emb_file = os.path.join(EMB_DIR, "passage.npy")
passages_file = os.path.join(EMB_DIR, "passages.json")

assert os.path.isfile(emb_file), f"Missing embedding file: {emb_file}"
assert os.path.isfile(passages_file), f"Missing passages file: {passages_file}"

embs = np.load(emb_file, mmap_mode="r")
passages = json.load(open(passages_file, "r"))

print("Embedding dir:", EMB_DIR)
print("passage.npy exists:", os.path.isfile(emb_file))
print("passages.json exists:", os.path.isfile(passages_file))
print("Embeddings shape:", embs.shape)
print("Num passages:", len(passages))
print("Example keys:", list(passages[0].keys()))

os.makedirs(os.path.join(BASE_DIR, "DATA", "KG", "graphs"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "configs", "kg_construct"), exist_ok=True)

Embedding dir: /content/drive/MyDrive/final_project/baseline/DATA/KG/emb/emb_2wikimultihopqa_dev2020wiki_1000_old
passage.npy exists: True
passages.json exists: True
Embeddings shape: (133890, 768)
Num passages: 133890
Example keys: ['title', 'passage', 'passage_id']


In [7]:
# Cell 7
# Write 2Wiki KG config
import os
import yaml
import torch

cfg_dir = os.path.join(BASE_DIR, "configs", "kg_construct")
cfg_2wiki = os.path.join(cfg_dir, "2wikimultihopqa_kg_construct.yml")

device = "cuda:0" if torch.cuda.is_available() else "cpu"
run_id = "2wikimultihopqa_dev2020wiki_1000_old"

args_dict = {
    "root_dir": ".",
    "method": "gpu",
    "run_id": run_id,
    "algo_params": {
        "n_neighbors": 20,
        "metric": "cosine",
        "batch_size": 100,
        "device": device,
    },
    "emb_file": f"DATA/KG/emb/emb_{run_id}/passage.npy",
    "passages_file": f"DATA/KG/emb/emb_{run_id}/passages.json",
}

with open(cfg_2wiki, "w") as f:
    yaml.dump(args_dict, f, sort_keys=False)

repo_graph_dir = os.path.join(BASE_DIR, "DATA", "KG", "graphs", f"graph_{run_id}")

print("Wrote config:", cfg_2wiki)
print("Preview:", yaml.safe_load(open(cfg_2wiki, "r")))
print("Repo-style graph dir:", repo_graph_dir)

Wrote config: /content/drive/MyDrive/final_project/baseline/configs/kg_construct/2wikimultihopqa_kg_construct.yml
Preview: {'root_dir': '.', 'method': 'gpu', 'run_id': '2wikimultihopqa_dev2020wiki_1000_old', 'algo_params': {'n_neighbors': 20, 'metric': 'cosine', 'batch_size': 100, 'device': 'cuda:0'}, 'emb_file': 'DATA/KG/emb/emb_2wikimultihopqa_dev2020wiki_1000_old/passage.npy', 'passages_file': 'DATA/KG/emb/emb_2wikimultihopqa_dev2020wiki_1000_old/passages.json'}
Repo-style graph dir: /content/drive/MyDrive/final_project/baseline/DATA/KG/graphs/graph_2wikimultihopqa_dev2020wiki_1000_old


In [8]:
# Cell 8
# Build KG and save it
import os
import json
import yaml
import pickle
import numpy as np

cfg_path = "./configs/kg_construct/2wikimultihopqa_kg_construct.yml"
args = load_config(cfg_path)

embs = np.load(os.path.join(args["root_dir"], args["emb_file"]))
print("Loaded embeddings:", embs.shape)

passages_data = json.load(open(os.path.join(args["root_dir"], args["passages_file"]), "r"))
print("Loaded passages:", len(passages_data))

G = get_kg_graph_gpu(embs=embs, **args["algo_params"])
print("Graph built.")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

updated_G = add_node_features(G, passages_data, embs)
print("Node features added.")

sample_node = 0
print("Sample node:", sample_node)
print("Sample node keys:", list(updated_G.nodes[sample_node].keys()))
print("Sample node title:", updated_G.nodes[sample_node]["title"])

repo_graph_dir = os.path.join(args["root_dir"], "DATA", "KG", "graphs", f"graph_{args['run_id']}")
os.makedirs(repo_graph_dir, exist_ok=True)

repo_graph_path = os.path.join(repo_graph_dir, "graph.gpickle")
repo_cfg_path = os.path.join(repo_graph_dir, "config.yml")

with open(repo_graph_path, "wb") as f:
    pickle.dump(updated_G, f, pickle.HIGHEST_PROTOCOL)

with open(repo_cfg_path, "w") as f:
    yaml.dump(args, f, sort_keys=False)

print("Repo-style graph saved to:", os.path.abspath(repo_graph_dir))
print("Saved KG path:", os.path.abspath(repo_graph_dir))

Loaded embeddings: (133890, 768)
Loaded passages: 133890


100%|██████████| 1339/1339 [01:13<00:00, 18.27it/s]


Graph built.
Nodes: 133890
Edges: 1887258


Adding node features: 100%|██████████| 133890/133890 [00:00<00:00, 615555.25it/s]


Node features added.
Sample node: 0
Sample node keys: ['title', 'passage', 'emb']
Sample node title: Calloway County High School
Repo-style graph saved to: /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline/DATA/KG/graphs/graph_2wikimultihopqa_dev2020wiki_1000_old
Saved KG path: /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline/DATA/KG/graphs/graph_2wikimultihopqa_dev2020wiki_1000_old


In [9]:
# Cell 9
# Verify saved files
import os
import yaml

cfg_path = os.path.join(BASE_DIR, "configs", "kg_construct", "2wikimultihopqa_kg_construct.yml")
args = yaml.safe_load(open(cfg_path, "r"))

repo_graph_dir = os.path.join(BASE_DIR, "DATA", "KG", "graphs", f"graph_{args['run_id']}")
repo_graph_path = os.path.join(repo_graph_dir, "graph.gpickle")
repo_cfg_path = os.path.join(repo_graph_dir, "config.yml")

print("Repo graph dir:", repo_graph_dir)
print("repo graph.gpickle exists:", os.path.isfile(repo_graph_path))
print("repo config.yml exists:", os.path.isfile(repo_cfg_path))

if os.path.isfile(repo_graph_path):
    print("graph.gpickle size (GB):", round(os.path.getsize(repo_graph_path) / (1024**3), 3))

print("Saved KG path:", os.path.abspath(repo_graph_dir))

Repo graph dir: /content/drive/MyDrive/final_project/baseline/DATA/KG/graphs/graph_2wikimultihopqa_dev2020wiki_1000_old
repo graph.gpickle exists: True
repo config.yml exists: True
graph.gpickle size (GB): 0.435
Saved KG path: /content/drive/MyDrive/final_project/baseline/DATA/KG/graphs/graph_2wikimultihopqa_dev2020wiki_1000_old
